# IT Incident Response Debate

**Pattern:** Agents discuss and challenge each other

Application, database, and network agents offer competing hypotheses about a fictional checkout outage. An incident lead separates evidence from assumptions.

Classroom simulation with fictional facts. Save an .env file beside this notebook with OPENAI_API_KEY=your_key. Install requirements.txt first.

In [ ]:
import os  # Read environment settings.
from dotenv import load_dotenv  # Load values from the .env file.
from langchain_openai import ChatOpenAI  # Use LangChain to call an OpenAI chat model.

load_dotenv()  # Load OPENAI_API_KEY from .env.
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)  # Share one model connection across role prompts.

def ask_agent(role, task, context):
    prompt = f"You are {role}. {task}\n\nCase facts and prior reports:\n{context}"  # Give this agent its role and handoff context.
    return model.invoke(prompt).content  # Return the agent's message.


## Synthetic case

Edit the sample details to explore how the agents respond.

In [ ]:
case = '10:05 AM checkout response time rose from 1 to 12 seconds. CPU is normal. Database connection pool is at 98%. Deployment occurred at 9:50 AM. Some users see timeouts.'  # Fictional case facts shared with the agents.
print(case)  # Inspect facts before running the workflow.


## Agent conversation

Each role is a short prompt. The coordinator receives reports from earlier roles where the workflow requires a handoff.

In [ ]:
goal = "Complete the defined task using the available evidence."  # Define the target.
plan = ask_agent("Workflow planner", "Plan short steps using only these configured roles: Application agent, Database agent, Network agent, Incident lead", case)  # Ask the LLM to plan.
print("PLAN:\n", plan)  # Show the plan before actions.
reports = []  # Store agent messages as observations and handoffs.
report = ask_agent('Application agent', 'Give one likely cause and evidence from the facts.', case)  # Run the Application agent role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Application agent ---\n", report)  # Make the conversation visible.
report = ask_agent('Database agent', 'Give one likely cause and evidence from the facts.', case)  # Run the Database agent role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Database agent ---\n", report)  # Make the conversation visible.
report = ask_agent('Network agent', 'Give one likely cause and evidence from the facts.', case)  # Run the Network agent role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Network agent ---\n", report)  # Make the conversation visible.
report = ask_agent('Incident lead', 'Compare views, point out disagreement, separate facts from hypotheses, and list next checks. Do not claim a confirmed cause.', case + "\n\nPrior agent messages:\n" + "\n".join(reports))  # Run the Incident lead role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Incident lead ---\n", report)  # Make the conversation visible.
observation = ask_agent("Observer", "Summarize the results and evidence from the agent messages.", case + "\n\nAgent messages:\n" + "\n".join(reports))  # Observe the actions.
print("\nOBSERVATION:\n", observation)
complete = False  # Track whether the goal has been met.
for attempt in range(2):  # Limit the workflow to two review rounds.
    status = ask_agent("Goal checker", "Reply COMPLETE if the goal is met; otherwise NEEDS_REVISION and one specific fix.", goal + "\n\n" + observation)  # Check the result.
    print(f"CHECK {attempt + 1}: {status}")
    if status.strip().upper().startswith("COMPLETE"):
        complete = True
        break  # Stop when the goal checker approves completion.
    observation = ask_agent("Workflow coordinator", "Apply the checker’s fix and return an improved result.", observation + "\n\nChecker: " + status)  # Iterate using feedback.
    print("ITERATION RESULT:\n", observation)
if not complete:
    print("Iteration limit reached. A person should review the result.")


## Plan, act, observe, iterate

The LLM first proposes a plan. The configured agents take actions by making role-specific calls and handing off messages. An observer summarizes the results. A goal checker decides whether to stop or request a revision. The coordinator may iterate up to two times, then asks for human review if the goal remains incomplete.

Agents discuss and challenge each other.